<a href="https://colab.research.google.com/github/StevenFAP1815/BioCLIP_Proyect/blob/main/03_bioclip_zeroshot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
print("hola")

hola


In [ ]:
import torch

if torch.cuda.is_available():
    device = "cuda"
    print("GPU disponible:", torch.cuda.get_device_name(0))
else:
    device = "cpu"
    print("No se detectó una GPU. Se utilizará CPU.")

print("Dispositivo seleccionado:", device)
print("Versión de PyTorch:", torch.__version__)

GPU disponible: Tesla T4
Dispositivo seleccionado: cuda
Versión de PyTorch: 2.11.0+cu128


# **1. Dependencias**

In [ ]:
%pip install -q open_clip_torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 64.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.5 MB/s eta 0:00:00


In [ ]:
import open_clip
import torch

print("OpenCLIP instalado correctamente")

OpenCLIP instalado correctamente


# **2. Carga de BioCLIP**

In [ ]:
import torch
import open_clip

MODEL_ID = "hf-hub:imageomics/bioclip"

# Cargar arquitectura, pesos y transformaciones
model, preprocess_train, preprocess_val = (
    open_clip.create_model_and_transforms(MODEL_ID)
)

# Cargar el tokenizador correspondiente
tokenizer = open_clip.get_tokenizer(MODEL_ID)

# Enviar el modelo a la GPU
model = model.to(device)
model.eval()

print("BioCLIP cargado correctamente")
print("Dispositivo:", next(model.parameters()).device)
print("Modelo:", MODEL_ID)

open_clip_config.json:   0%|          | 0.00/469 [00:00<?, ?B/s]

open_clip_pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  599MB            

open_clip_pytorch_model.bin: downloading bytes:           |  0.00B            

BioCLIP cargado correctamente
Dispositivo: cuda:0
Modelo: hf-hub:imageomics/bioclip


## ***Verificación del modelo***

In [ ]:
# Comprobar que el modelo puede procesar una imagen
from PIL import Image
import torch

# Imagen sintética RGB de 224 × 224 píxeles
dummy_image = Image.new("RGB", (224, 224), color=(120, 160, 100))

# Aplicar el preprocesamiento de validación
image_tensor = preprocess_val(dummy_image).unsqueeze(0).to(device)

with torch.inference_mode():
    image_features = model.encode_image(image_tensor)

print("Forma del tensor de entrada:", tuple(image_tensor.shape))
print("Forma de las características:", tuple(image_features.shape))
print("¿Todos los valores son finitos?:", torch.isfinite(image_features).all().item())

Forma del tensor de entrada: (1, 3, 224, 224)
Forma de las características: (1, 512)
¿Todos los valores son finitos?: True


# **3. Recuperar las categorías**

Let's check if the directory and the file actually exist. This will help us diagnose why the `FileNotFoundError` occurred.

In [ ]:
import os
from pathlib import Path
from google.colab import drive

# 1. Asegurar que Google Drive esté montado
drive.mount('/content/drive')

# Ruta base donde buscar (puedes ajustar a /content/drive/MyDrive para todo tu Drive)
search_base = Path("/content/drive/MyDrive/dataset-LZUPSD")

print("=" * 70)
print(f"🔍 BUSCANDO ARCHIVOS 'config.json' Y SIMILARES EN: {search_base}")
print("=" * 70)

found_files = []

if search_base.exists():
    # Recorrer carpetas en busca de config.json o cualquier archivo .json
    for root, dirs, files in os.walk(search_base):
        for file in files:
            # Buscar cualquier archivo .json o que contenga la palabra 'config'
            if file.endswith(".json") or "config" in file.lower():
                full_path = Path(root) / file
                found_files.append(full_path)

    if found_files:
        print(f"✅ Se encontraron {len(found_files)} archivo(s) de configuración/JSON:\n")
        for path in found_files:
            print(f"📄 Archivo: {path.name}")
            print(f"   └─ Ruta completa para tu script: Path('{path}')\n")
    else:
        print("❌ No se encontró ningún archivo 'config.json' en esta ruta.")
        print("👉 Si aún no has creado o generado el archivo de configuración, asegúrate de guardar 'config.json' antes de intentar abrirlo.")
else:
    print(f"❌ La carpeta base '{search_base}' no existe.")

Mounted at /content/drive
🔍 BUSCANDO ARCHIVOS 'config.json' Y SIMILARES EN: /content/drive/MyDrive/dataset-LZUPSD
✅ Se encontraron 2 archivo(s) de configuración/JSON:

📄 Archivo: config.json
   └─ Ruta completa para tu script: Path('/content/drive/MyDrive/dataset-LZUPSD/prepared_1/config.json')

📄 Archivo: config.json
   └─ Ruta completa para tu script: Path('/content/drive/MyDrive/dataset-LZUPSD/prepared_3/config.json')



In [ ]:
import json
from pathlib import Path

# Ruta donde guardamos la preparación del dataset
OUTPUT_DIR = Path("/content/drive/MyDrive/dataset-LZUPSD/prepared_3")

# Cargar configuración
with open(OUTPUT_DIR / "config.json", "r", encoding="utf-8") as file:
    config = json.load(file)

print("Configuración cargada correctamente")
print("Número de clases:", config["num_classes"])
print("Semilla:", config["random_state"])
print("Train:", config["train_size"])
print("Validation:", config["validation_size"])
print("Test:", config["test_size"])

Configuración cargada correctamente
Número de clases: 88
Semilla: 42
Train: 0.7
Validation: 0.15
Test: 0.15


In [ ]:
class_to_id = config["class_to_id"]

print("Número de categorías:", len(class_to_id))

print("\nPrimeras 10 categorías:")
for class_name, class_id in list(class_to_id.items())[:10]:
    print(f"{class_id}: {class_name}")

Número de categorías: 88

Primeras 10 categorías:
0: Achnatherum inebrians
1: Achnatherum splendens
2: Agriophyllum squarrosum
3: Agropyron cristatum
4: Agropyron elongatum
5: Agropyron mongolicum
6: Amorpha fruticosa
7: Amygdalus mongolica
8: Anemone rivularis
9: Apocynum pictum


# **4. Crear la lista de catageória**
***Primero vamos a convertir el diccionario class_to_id en una lista ordenada por class_id. Esto es importante porque las posiciones de la lista deben coincidir con los identificadores de nuestras clases.***

In [ ]:
# Crear lista de categorías ordenada por class_id
id_to_class = {
    class_id: class_name
    for class_name, class_id in class_to_id.items()
}

class_names = [
    id_to_class[class_id]
    for class_id in sorted(id_to_class)
]

print("Número de categorías:", len(class_names))

print("\nPrimeras 10 categorías:")
for class_id, class_name in enumerate(class_names[:10]):
    print(f"{class_id}: {class_name}")

print("\nÚltimas 5 categorías:")
for class_id, class_name in enumerate(class_names[-5:], start=83):
    print(f"{class_id}: {class_name}")

Número de categorías: 88

Primeras 10 categorías:
0: Achnatherum inebrians
1: Achnatherum splendens
2: Agriophyllum squarrosum
3: Agropyron cristatum
4: Agropyron elongatum
5: Agropyron mongolicum
6: Amorpha fruticosa
7: Amygdalus mongolica
8: Anemone rivularis
9: Apocynum pictum

Últimas 5 categorías:
83: Trifolium repens
84: Triticale
85: Vicia sativa
86: Vicia villosa
87: Zygophyllum xanthoxylon


# **5. Crear los prompts de contexto**

In [ ]:
"""
La presición de BioCLIP decae en un 10 a 7% al usar palabras en español
por lo que para mantener un score alto usaremos frases en inglés
"""

# Plantilla de texto para la evaluación Zero-Shot
TEXT_TEMPLATE = "a photo of a seed of {}"

text_prompts = [
    TEXT_TEMPLATE.format(class_name)
    for class_name in class_names
]

print("Número de prompts:", len(text_prompts))

print("\nPrimeros 5 prompts:")
for prompt in text_prompts[:5]:
    print(prompt)

Número de prompts: 88

Primeros 5 prompts:
a photo of a seed of Achnatherum inebrians
a photo of a seed of Achnatherum splendens
a photo of a seed of Agriophyllum squarrosum
a photo of a seed of Agropyron cristatum
a photo of a seed of Agropyron elongatum


# **6. Generar lo embbedings**

In [ ]:
# Tokenizar los 88 prompts
text_tokens = tokenizer(text_prompts).to(device)

print("Forma de los tokens:", tuple(text_tokens.shape))

Forma de los tokens: (88, 77)


In [ ]:
# Obtener embeddings de texto
with torch.inference_mode():
    text_features = model.encode_text(text_tokens)

print("Forma de los embeddings:", tuple(text_features.shape))
print("¿Todos los valores son finitos?:", torch.isfinite(text_features).all().item())

Forma de los embeddings: (88, 512)
¿Todos los valores son finitos?: True


## **6.1 Normalizar los embeddings**

In [ ]:
# Normalizar embeddings de texto
text_features = text_features / text_features.norm(
    dim=-1,
    keepdim=True
)

print("Forma después de normalizar:", tuple(text_features.shape))

# Verificar las normas
text_norms = text_features.norm(dim=-1)

print("Norma mínima:", text_norms.min().item())
print("Norma máxima:", text_norms.max().item())

Forma después de normalizar: (88, 512)
Norma mínima: 0.9999998807907104
Norma máxima: 1.0000001192092896


## **6.2 Cargar una imagen de prueba**

imagen → preprocess → embedding visual → normalización → similitud con 88 textos → predicción

In [ ]:
# Definir la ruta base del dataset, que es donde se encuentran las imágenes.
DATASET_DIR = Path("/content/drive/MyDrive/dataset-LZUPSD")

print(f"Ruta base del dataset establecida a: {DATASET_DIR}")

Ruta base del dataset establecida a: /content/drive/MyDrive/dataset-LZUPSD


In [ ]:
from pathlib import Path
import pandas as pd

# Rutas correctas
DATASET_DIR = Path("/content/Seed_dataset_extracted/Seed dataset")
OUTPUT_DIR = Path("/content/drive/MyDrive/dataset-LZUPSD/prepared_3")

# Cargar test
df_test = pd.read_csv(OUTPUT_DIR / "test.csv")

# Reconstruir las rutas usando el directorio donde está EXTRAÍDO el dataset
df_test["image_path"] = df_test["image_path"].apply(
    lambda path: str(DATASET_DIR / path)
)

# Seleccionar una imagen
sample = df_test.iloc[0]

image_path = sample["image_path"]
true_class_id = int(sample["class_id"])
true_class_name = sample["class_name"]

print("Imagen:", image_path)
print("Existe:", Path(image_path).exists())
print("Clase real:", true_class_id)
print("Nombre:", true_class_name)

Imagen: /content/Seed_dataset_extracted/Seed dataset/Medicago lupulina/changed700.jpg
Existe: False
Clase real: 62
Nombre: Medicago lupulina


In [ ]:
import os
from pathlib import Path
from google.colab import drive

# 1. Montar Google Drive
drive.mount('/content/drive')

# 2. Definir la ruta raíz de Google Drive
drive_root = Path("/content/drive/MyDrive")

print("=" * 75)
print("📂 EXPLORACIÓN COMPLETA DE CARPETAS Y SUBCARPETAS EN GOOGLE DRIVE")
print("=" * 75)

if drive_root.exists():
    print(f"Ruta raíz base: {drive_root}\n")

    # Recorrer recursivamente todos los niveles del directorio
    for root, dirs, files in os.walk(drive_root):
        current_path = Path(root)

        # Ignorar carpetas ocultas del sistema (las que empiezan con '.')
        dirs[:] = [d for d in dirs if not d.startswith('.')]

        # Calcular el nivel de profundidad respecto a MyDrive
        depth = len(current_path.relative_to(drive_root).parts)
        indent = "│   " * depth

        if depth == 0:
            print(f"📁 MyDrive/ ({current_path})")
        else:
            folder_name = current_path.name
            print(f"{indent}├── 📁 {folder_name}/")
            print(f"{indent}│   └─ Ruta: {current_path}")

    print("\n" + "=" * 75)
    print("✅ Recorrido finalizado exitosamente.")
else:
    print("❌ No se pudo acceder a Google Drive. Revisa la conexión del montaje.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📂 EXPLORACIÓN COMPLETA DE CARPETAS Y SUBCARPETAS EN GOOGLE DRIVE
Ruta raíz base: /content/drive/MyDrive

📁 MyDrive/ (/content/drive/MyDrive)
│   ├── 📁 Colab Notebooks/
│   │   └─ Ruta: /content/drive/MyDrive/Colab Notebooks
│   ├── 📁 dataset-LZUPSD/
│   │   └─ Ruta: /content/drive/MyDrive/dataset-LZUPSD
│   │   ├── 📁 prepared_1/
│   │   │   └─ Ruta: /content/drive/MyDrive/dataset-LZUPSD/prepared_1
│   │   ├── 📁 prepared_3/
│   │   │   └─ Ruta: /content/drive/MyDrive/dataset-LZUPSD/prepared_3

✅ Recorrido finalizado exitosamente.


In [ ]:
from pathlib import Path

PREPARED_DIR = Path("/content/drive/MyDrive/dataset-LZUPSD/prepared_3")

print("📂 Contenido de prepared_3:")
for item in sorted(PREPARED_DIR.iterdir()):
    print(" -", item.name)

📂 Contenido de prepared_3:
 - config.json
 - excluded_images.csv
 - test.csv
 - train.csv
 - val.csv


In [ ]:
import pandas as pd
from pathlib import Path

PREPARED_DIR = Path("/content/drive/MyDrive/dataset-LZUPSD/prepared_3")

df_test = pd.read_csv(PREPARED_DIR / "test.csv")

print("Columnas:")
print(df_test.columns.tolist())

print("\nPrimeras 5 rutas guardadas en test.csv:")
for path in df_test["image_path"].head():
    print(repr(path))

Columnas:
['image_path', 'class_name', 'class_id']

Primeras 5 rutas guardadas en test.csv:
'Medicago\xa0lupulina/changed700.jpg'
'Lolium perenne/changed485.jpg'
'Astragalus melilotoides/changed3635.jpg'
'Ligularia virgaurea/changed4428.jpg'
'Caragana liouana/changed8.jpg'


In [ ]:
from pathlib import Path

# Buscar dónde está realmente la carpeta de Medicago
search_roots = [
    Path("/content"),
    Path("/content/drive/MyDrive/dataset-LZUPSD")
]

for root in search_roots:
    print(f"\n🔎 Buscando dentro de: {root}")

    if root.exists():
        matches = list(root.rglob("*Medicago*"))

        for match in matches[:20]:
            print("  ", repr(str(match)))


🔎 Buscando dentro de: /content

🔎 Buscando dentro de: /content/drive/MyDrive/dataset-LZUPSD


In [ ]:
from pathlib import Path

DATASET_ROOT = Path("/content/drive/MyDrive/dataset-LZUPSD")

print("📦 Archivos dentro de dataset-LZUPSD:")

for item in DATASET_ROOT.iterdir():
    print(f" - {item.name}")

📦 Archivos dentro de dataset-LZUPSD:
 - Seed dataset.zip
 - prepared_1
 - prepared_3


In [ ]:
from pathlib import Path
import zipfile

ZIP_PATH = Path("/content/drive/MyDrive/dataset-LZUPSD/Seed dataset.zip")
EXTRACT_DIR = Path("/content/Seed_dataset_extracted")

print("ZIP existe:", ZIP_PATH.exists())

# Crear directorio de extracción
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

# Extraer
with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)

print("\nExtracción completada.")
print("Contenido de /content/Seed_dataset_extracted:")

for item in EXTRACT_DIR.iterdir():
    print(" -", repr(item.name))

ZIP existe: True

Extracción completada.
Contenido de /content/Seed_dataset_extracted:
 - 'Seed dataset'


In [ ]:
from pathlib import Path

DATASET_DIR = Path("/content/Seed_dataset_extracted/Seed dataset")

print("¿Existe DATASET_DIR?:", DATASET_DIR.exists())

print("\nCarpetas que contienen 'Medicago':")
for folder in DATASET_DIR.iterdir():
    if "Medicago" in folder.name:
        print("Nombre:", repr(folder.name))
        print("Ruta:", repr(str(folder)))

¿Existe DATASET_DIR?: True

Carpetas que contienen 'Medicago':
Nombre: 'Medicago\xa0lupulina'
Ruta: '/content/Seed_dataset_extracted/Seed dataset/Medicago\xa0lupulina'
Nombre: 'Medicago sativa'
Ruta: '/content/Seed_dataset_extracted/Seed dataset/Medicago sativa'


In [ ]:
import pandas as pd
from pathlib import Path

PREPARED_DIR = Path("/content/drive/MyDrive/dataset-LZUPSD/prepared_3")
DATASET_DIR = Path("/content/Seed_dataset_extracted/Seed dataset")

# Cargar test de prepared_3
df_test = pd.read_csv(PREPARED_DIR / "test.csv")

# Tomar la primera imagen
relative_path = df_test.iloc[0]["image_path"]
image_path = DATASET_DIR / relative_path

print("Ruta relativa:")
print(repr(relative_path))

print("\nRuta completa:")
print(repr(str(image_path)))

print("\n¿Existe la imagen?:", image_path.exists())

print("\nClase:")
print(df_test.iloc[0]["class_name"])

Ruta relativa:
'Medicago\xa0lupulina/changed700.jpg'

Ruta completa:
'/content/Seed_dataset_extracted/Seed dataset/Medicago\xa0lupulina/changed700.jpg'

¿Existe la imagen?: True

Clase:
Medicago lupulina


## **6.3 Procesar la imagen**

In [ ]:
from PIL import Image

image = Image.open(image_path).convert("RGB")

image_tensor = preprocess_val(image).unsqueeze(0).to(device)

print("Tamaño original:", image.size)
print("Tensor de entrada:", tuple(image_tensor.shape))

Tamaño original: (192, 272)
Tensor de entrada: (1, 3, 224, 224)


In [ ]:
with torch.inference_mode():
    image_features = model.encode_image(image_tensor)

print("Embedding visual:", tuple(image_features.shape))
print("¿Valores finitos?:", torch.isfinite(image_features).all().item())

Embedding visual: (1, 512)
¿Valores finitos?: True


In [ ]:
image_features = image_features / image_features.norm(dim=-1, keepdim=True)

print("Norma del embedding visual:", image_features.norm(dim=-1).item())

Norma del embedding visual: 1.0


# **7. Predicción con Zero-shot**

In [ ]:
with torch.inference_mode():
    similarities = image_features @ text_features.T

print("Forma de similitudes:", tuple(similarities.shape))
print("Similitud máxima:", similarities.max().item())

predicted_class_id = similarities.argmax(dim=-1).item()
predicted_class_name = class_names[predicted_class_id]

print("\nClase real:", true_class_name)
print("Clase predicha:", predicted_class_name)
print("ID predicho:", predicted_class_id)

Forma de similitudes: (1, 88)
Similitud máxima: 0.2158702313899994

Clase real: Medicago lupulina
Clase predicha: Trifolium repens
ID predicho: 83


Predicción mala

In [ ]:
top_k = 5

top_values, top_indices = similarities[0].topk(top_k)

print(f"Top-{top_k} predicciones:\n")

for rank, (score, idx) in enumerate(
    zip(top_values.tolist(), top_indices.tolist()), start=1
):
    print(
        f"{rank}. {class_names[idx]} "
        f"(ID={idx}, similitud={score:.6f})"
    )

Top-5 predicciones:

1. Trifolium repens (ID=83, similitud=0.215870)
2. Elaeagnus angustifolia (ID=35, similitud=0.210865)
3. Calligonum mongolicum Turcz (ID=21, similitud=0.203795)
4. Lolium perenne (ID=58, similitud=0.197911)
5. Salsola tragus (ID=77, similitud=0.195470)


# **8. Construir DataLoader**

In [ ]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import pandas as pd
from pathlib import Path

class SeedTestDataset(Dataset):
    def __init__(self, dataframe, dataset_dir, preprocess):
        self.dataframe = dataframe.reset_index(drop=True)
        self.dataset_dir = Path(dataset_dir)
        self.preprocess = preprocess

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]

        image_path = self.dataset_dir / row["image_path"]

        image = Image.open(image_path).convert("RGB")
        image = self.preprocess(image)

        return image, int(row["class_id"])


# Dataset de test
test_dataset = SeedTestDataset(
    dataframe=df_test,
    dataset_dir=DATASET_DIR,
    preprocess=preprocess_val
)

# DataLoader
test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Imágenes de test:", len(test_dataset))
print("Número de batches:", len(test_loader))

Imágenes de test: 674
Número de batches: 22


# **9. Extracción de embeddings visuales para todo el test**

In [ ]:
import torch

all_image_features = []
all_labels = []

with torch.inference_mode():
    for images, labels in test_loader:
        images = images.to(device, non_blocking=True)

        features = model.encode_image(images)

        # Normalización L2
        features = features / features.norm(dim=-1, keepdim=True)

        all_image_features.append(features.cpu())
        all_labels.append(labels)

# Unir todos los batches
all_image_features = torch.cat(all_image_features, dim=0)
all_labels = torch.cat(all_labels, dim=0)

print("Embeddings visuales:", tuple(all_image_features.shape))
print("Etiquetas:", tuple(all_labels.shape))

print(
    "¿Todos los embeddings son finitos?:",
    torch.isfinite(all_image_features).all().item()
)

print(
    "Norma mínima:",
    all_image_features.norm(dim=-1).min().item()
)

print(
    "Norma máxima:",
    all_image_features.norm(dim=-1).max().item()
)

Embeddings visuales: (674, 512)
Etiquetas: (674,)
¿Todos los embeddings son finitos?: True
Norma mínima: 0.9999998211860657
Norma máxima: 1.0000001192092896


# **10. Calcular la similitud entre las 674 imágenes y las 88 clases**

In [ ]:
# Mover los embeddings de texto a CPU para trabajar con los embeddings visuales
text_features_cpu = text_features.cpu()

# Similitud imagen-texto
all_similarities = all_image_features @ text_features_cpu.T

print("Matriz de similitud:", tuple(all_similarities.shape))
print("¿Todos los valores son finitos?:", torch.isfinite(all_similarities).all().item())
print("Similitud mínima:", all_similarities.min().item())
print("Similitud máxima:", all_similarities.max().item())

Matriz de similitud: (674, 88)
¿Todos los valores son finitos?: True
Similitud mínima: -0.07991521805524826
Similitud máxima: 0.305125892162323


# **11. Obtener la clase predicha para cada imagen tomando la clase con mayor similitud**

In [ ]:
# Obtener la clase con mayor similitud para cada imagen
predicted_ids = all_similarities.argmax(dim=1)

print("Predicciones:", predicted_ids.shape)
print("Primeras 10 predicciones:", predicted_ids[:10].tolist())
print("Rango de IDs predichos:",
      predicted_ids.min().item(),
      "a",
      predicted_ids.max().item())

Predicciones: torch.Size([674])
Primeras 10 predicciones: [83, 83, 35, 19, 62, 47, 83, 35, 61, 17]
Rango de IDs predichos: 0 a 87


## **12. Comparar las predicciones contra las etiquetas reales y calcular la accuracy del baseline zero-shot**

In [ ]:
from sklearn.metrics import accuracy_score

# Convertir a NumPy
y_true = all_labels.numpy()
y_pred = predicted_ids.numpy()

# Accuracy
accuracy = accuracy_score(y_true, y_pred)

print(f"Accuracy Zero-Shot: {accuracy:.4f}")
print(f"Accuracy Zero-Shot: {accuracy * 100:.2f}%")

Accuracy Zero-Shot: 0.0208
Accuracy Zero-Shot: 2.08%


# **13. Métricas completas**

In [ ]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

# Métricas globales
precision_macro = precision_score(
    y_true, y_pred,
    average="macro",
    zero_division=0
)

recall_macro = recall_score(
    y_true, y_pred,
    average="macro",
    zero_division=0
)

f1_macro = f1_score(
    y_true, y_pred,
    average="macro",
    zero_division=0
)

f1_weighted = f1_score(
    y_true, y_pred,
    average="weighted",
    zero_division=0
)

print(f"Accuracy:          {accuracy:.4f}")
print(f"Precision macro:   {precision_macro:.4f}")
print(f"Recall macro:      {recall_macro:.4f}")
print(f"F1 macro:         {f1_macro:.4f}")
print(f"F1 weighted:      {f1_weighted:.4f}")

Accuracy:          0.0208
Precision macro:   0.0105
Recall macro:      0.0219
F1 macro:         0.0119
F1 weighted:      0.0123


In [ ]:
from sklearn.metrics import confusion_matrix
import numpy as np

# Matriz de confusión
cm = confusion_matrix(
    y_true,
    y_pred,
    labels=np.arange(len(class_names))
)

print("Dimensión de la matriz de confusión:", cm.shape)
print("Total de muestras:", cm.sum())
print("Aciertos:", np.trace(cm))

Dimensión de la matriz de confusión: (88, 88)
Total de muestras: 674
Aciertos: 14


In [ ]:
# Accuracy por clase
class_accuracy = []

for class_id, class_name in enumerate(class_names):
    total = cm[class_id].sum()
    correct = cm[class_id, class_id]

    accuracy_class = correct / total if total > 0 else 0.0

    class_accuracy.append({
        "class_id": class_id,
        "class_name": class_name,
        "correct": correct,
        "total": total,
        "accuracy": accuracy_class
    })

# Mostrar las clases ordenadas por accuracy
class_accuracy_sorted = sorted(
    class_accuracy,
    key=lambda x: x["accuracy"],
    reverse=True
)

for row in class_accuracy_sorted:
    print(
        f"{row['class_id']:2d} | "
        f"{row['class_name']:<40} | "
        f"{row['correct']:2d}/{row['total']:2d} | "
        f"{row['accuracy']:.2%}"
    )

69 | Pisum sativum L                          |  3/ 7 | 42.86%
 7 | Amygdalus mongolica                      |  2/ 6 | 33.33%
82 | Thlaspi arvense                          |  2/ 7 | 28.57%
 1 | Achnatherum splendens                    |  1/ 6 | 16.67%
19 | Bromus inermis                           |  1/ 6 | 16.67%
35 | Elaeagnus angustifolia                   |  1/ 6 | 16.67%
53 | Lespedeza bicolor                        |  1/ 8 | 12.50%
58 | Lolium perenne                           |  1/ 9 | 11.11%
71 | Poa annua                                |  1/12 | 8.33%
61 | Medicago sativa                          |  1/16 | 6.25%
 0 | Achnatherum inebrians                    |  0/ 6 | 0.00%
 2 | Agriophyllum squarrosum                  |  0/13 | 0.00%
 3 | Agropyron cristatum                      |  0/ 8 | 0.00%
 4 | Agropyron elongatum                      |  0/ 6 | 0.00%
 5 | Agropyron mongolicum                     |  0/ 7 | 0.00%
 6 | Amorpha fruticosa                        |  0/ 6 | 0.00%


In [ ]:
from collections import Counter

# Contar cuántas veces fue predicha cada clase
prediction_counts = Counter(y_pred.tolist())

print("Clase | Nombre | Predicciones")

for class_id, count in prediction_counts.most_common():
    print(
        f"{class_id:2d} | "
        f"{class_names[class_id]:<40} | "
        f"{count:3d}"
    )

Clase | Nombre | Predicciones
35 | Elaeagnus angustifolia                   | 113
83 | Trifolium repens                         | 110
58 | Lolium perenne                           |  84
62 | Medicago lupulina                        |  37
44 | Halostachys caspica                      |  32
 7 | Amygdalus mongolica                      |  30
69 | Pisum sativum L                          |  24
47 | Hippophae rhamnoides                     |  21
82 | Thlaspi arvense                          |  21
70 | Plantago major                           |  20
77 | Salsola tragus                           |  14
71 | Poa annua                                |  14
21 | Calligonum mongolicum Turcz              |  13
 1 | Achnatherum splendens                    |  13
49 | Hyoscyamus niger                         |  10
19 | Bromus inermis                           |   9
61 | Medicago sativa                          |   9
60 | Lycium ruthenicum                        |   9
65 | Nitraria tangutorum          

In [ ]:
# Top-3 predictions
top3_predictions = torch.topk(
    all_similarities,
    k=3,
    dim=1
).indices

# Comprobar si la etiqueta real está entre las 3 predicciones
top3_correct = (
    top3_predictions == all_labels.unsqueeze(1)
).any(dim=1)

top3_accuracy = top3_correct.float().mean().item()

print(f"Top-3 Accuracy Zero-Shot: {top3_accuracy:.4f}")
print(f"Top-3 Accuracy Zero-Shot: {top3_accuracy * 100:.2f}%")

Top-3 Accuracy Zero-Shot: 0.0564
Top-3 Accuracy Zero-Shot: 5.64%


# **13. Revisar ejemplos concretos de aciertos y errores junto con sus similitudes**

In [ ]:
# Mostrar ejemplos de predicciones correctas
correct_indices = torch.where(predicted_ids == all_labels)[0]

print("Número de aciertos:", len(correct_indices))
print()

for idx in correct_indices[:10]:
    true_id = all_labels[idx].item()
    pred_id = predicted_ids[idx].item()
    similarity = all_similarities[idx, pred_id].item()

    print(
        f"Índice: {idx.item():3d} | "
        f"Real: {class_names[true_id]} | "
        f"Predicha: {class_names[pred_id]} | "
        f"Similitud: {similarity:.4f}"
    )

Número de aciertos: 14

Índice:  23 | Real: Amygdalus mongolica | Predicha: Amygdalus mongolica | Similitud: 0.2412
Índice:  46 | Real: Pisum sativum L | Predicha: Pisum sativum L | Similitud: 0.2087
Índice:  47 | Real: Amygdalus mongolica | Predicha: Amygdalus mongolica | Similitud: 0.2237
Índice:  50 | Real: Thlaspi arvense | Predicha: Thlaspi arvense | Similitud: 0.1946
Índice:  91 | Real: Pisum sativum L | Predicha: Pisum sativum L | Similitud: 0.1920
Índice: 246 | Real: Pisum sativum L | Predicha: Pisum sativum L | Similitud: 0.2020
Índice: 391 | Real: Elaeagnus angustifolia | Predicha: Elaeagnus angustifolia | Similitud: 0.2264
Índice: 440 | Real: Medicago sativa | Predicha: Medicago sativa | Similitud: 0.2154
Índice: 521 | Real: Lolium perenne | Predicha: Lolium perenne | Similitud: 0.2309
Índice: 527 | Real: Bromus inermis | Predicha: Bromus inermis | Similitud: 0.2813


In [ ]:
# Obtener los errores
incorrect_indices = torch.where(predicted_ids != all_labels)[0]

# Confianza = similitud de la clase predicha
error_similarities = all_similarities[
    incorrect_indices,
    predicted_ids[incorrect_indices]
]

# Ordenar errores de mayor a menor similitud
sorted_order = torch.argsort(error_similarities, descending=True)

print("Número de errores:", len(incorrect_indices))
print()

for position in sorted_order[:20]:
    idx = incorrect_indices[position].item()

    true_id = all_labels[idx].item()
    pred_id = predicted_ids[idx].item()
    similarity = all_similarities[idx, pred_id].item()

    print(
        f"Índice: {idx:3d} | "
        f"Real: {class_names[true_id]} | "
        f"Predicha: {class_names[pred_id]} | "
        f"Similitud: {similarity:.4f}"
    )

Número de errores: 660

Índice: 604 | Real: Elymus nutans | Predicha: Bromus inermis | Similitud: 0.3051
Índice: 423 | Real: Clematis fruticosa | Predicha: Trifolium repens | Similitud: 0.3004
Índice: 392 | Real: Setaria viridis | Predicha: Trifolium repens | Similitud: 0.2943
Índice: 562 | Real: Triticale | Predicha: Elaeagnus angustifolia | Similitud: 0.2928
Índice: 271 | Real: Kochia scoparia | Predicha: Trifolium repens | Similitud: 0.2906
Índice: 407 | Real: Festuca rubra | Predicha: Calligonum mongolicum Turcz | Similitud: 0.2896
Índice: 501 | Real: Triticale | Predicha: Elaeagnus angustifolia | Similitud: 0.2882
Índice: 379 | Real: Calligonum mongolicum | Predicha: Elaeagnus angustifolia | Similitud: 0.2872
Índice: 472 | Real: Atriplex sibirica | Predicha: Elaeagnus angustifolia | Similitud: 0.2863
Índice: 640 | Real: Elymus sibiricus | Predicha: Achnatherum splendens | Similitud: 0.2862
Índice:  64 | Real: Ligularia virgaurea | Predicha: Lolium perenne | Similitud: 0.2799
Índic

# **14. Guardar los resultados del Baseline**

In [ ]:
# Guardar resultados del baseline zero-shot

df_results = df_test.copy()

df_results["predicted_id"] = y_pred
df_results["predicted_class"] = [
    class_names[class_id]
    for class_id in y_pred
]

df_results["correct"] = (
    df_results["class_id"] == df_results["predicted_id"]
)

df_results["similarity"] = all_similarities[
    torch.arange(len(y_pred)),
    predicted_ids
].numpy()

# Guardar en Drive
results_path = PREPARED_DIR / "zeroshot_baseline_results.csv"
df_results.to_csv(results_path, index=False)

print("Resultados guardados en:")
print(results_path)

print()
print("Filas:", len(df_results))
print("Columnas:", list(df_results.columns))

Resultados guardados en:
/content/drive/MyDrive/dataset-LZUPSD/prepared_3/zeroshot_baseline_results.csv

Filas: 674
Columnas: ['image_path', 'class_name', 'class_id', 'predicted_id', 'predicted_class', 'correct', 'similarity']


# **PARTE 2: Alternativas para mejorar la predicción**

***Podría mejorar un poco mas modificando algunas cosas, pero cerraremos aqui, su rendimiento fue muy bajo***